# 5. Reranking the Candidate Generator's Slates

`RR_4` trains and evaluates on **logged MIND slates**. This notebook instead trains the
same `+metadata` reranker on the **candidate generator's training slates** and evaluates it
on the generator's **validation slates**.

Protocol:

- **Training candidates**: the generator's Top-500 list for sampled training impressions.
  Candidates that were never shown have no label, so they are used as negatives (0).
  Clicks are joined from `labels_train`; the slate is capped with all clicks kept.
- **Tabular features**: generated candidates mostly have no row in
  `ranking_features_*`, so the causal timeline is replayed to build the same 50 features
  for every candidate. Replayed features are cached.
- **Validation**: every generated candidate is scored. Metrics come from
  `evaluate_retrieved_candidates`: unshown retrieved articles are unknown, every known click
  stays in the denominator, and `candidate_recall` is the ceiling for the ranking metrics.
- **Model**: only the final architecture and loss chosen in `RR_4` are retrained
  (`+metadata`, `softmax+lambda_ndcg`), not the whole architecture/loss grid.

**Prerequisite:** `recommendations_train_top500.parquet` must exist. Run
`5_01_Candidate_Generation/4_combined_models/CG_content_two_tower_rolling_popularity.ipynb`
once with `TARGET_SPLIT = "train"`, then restore it to `"validation"`.


## 1.1 Load Packages


In [1]:
import json
import random
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn


## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. Candidate
files are read from the candidate-generation artifact folder. Replayed features are written
under `5_02_Reranking/artifacts/branch_interaction_generated` so the logged-slate `RR_4`
artifacts and the pipeline evaluation notebook are left untouched.

Set `RUN_TAG` to a run name (for example `"train10000"`) to write a separate output folder
when changing `TRAIN_IMPRESSIONS`. The default `None` keeps the legacy path used by the
5,000-impression run, so existing checkpoints and scores are not overwritten.


In [2]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path("/Users/numkoos/Downloads/BT4222-Group-6-")
    DATA_ROOT = Path("/Users/numkoos/Downloads/v3-mindsmall")
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = next(
            p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
        )
    if not DATA_ROOT.exists():
        DATA_ROOT = PROJECT_ROOT

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# Candidate-generator output for this project
CANDIDATE_ARTIFACTS = (
    DATA_ROOT
    / "5_Models"
    / "5_01_Candidate_Generation"
    / "artifacts"
    / "popular_niche_combined"
)
if not CANDIDATE_ARTIFACTS.exists():
    CANDIDATE_ARTIFACTS = (
        PROJECT_ROOT
        / "5_Models"
        / "5_01_Candidate_Generation"
        / "artifacts"
        / "popular_niche_combined"
    )

GENERATED_TRAIN = CANDIDATE_ARTIFACTS / "recommendations_train_top450.parquet"
GENERATED_VALIDATION = CANDIDATE_ARTIFACTS / "recommendations_validation_top450.parquet"
GENERATED_TEST = CANDIDATE_ARTIFACTS / "recommendations_test_top450.parquet"

# The replay helpers expect <root>/1_Data_Preparation/processed
REPLAY_ROOT = (
    DATA_ROOT
    if (DATA_ROOT / "1_Data_Preparation" / "processed").exists()
    else PROJECT_ROOT
)
PROCESSED_ROOT = REPLAY_ROOT / "1_Data_Preparation" / "processed"

sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from newsrec.common import DEFAULT_SETTINGS
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.ranking import TemporalState, build_features, feature_schema
from newsrec.representations import Bundle
from rr_evaluation import evaluate_retrieved_candidates

if not GENERATED_TRAIN.exists():
    raise FileNotFoundError(
        "Generated training candidates are missing. Run "
        "5_01_Candidate_Generation/4_combined_models/CG_content_two_tower_rolling_popularity.ipynb "
        "once with TARGET_SPLIT='train' to write recommendations_train_top500.parquet."
    )
if not GENERATED_VALIDATION.exists():
    raise FileNotFoundError(f"Missing generated validation candidates: {GENERATED_VALIDATION}")

print("Project root:", PROJECT_ROOT)
print("Replay root:", REPLAY_ROOT)
print("Train candidates:", GENERATED_TRAIN)
print("Validation candidates:", GENERATED_VALIDATION)



Project root: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-
Replay root: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-
Train candidates: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\popular_niche_combined\recommendations_train_top450.parquet
Validation candidates: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\popular_niche_combined\recommendations_validation_top450.parquet


## 1.3 Preview Generated Candidates

One row per `(impression_uid, candidate)`, with `cutoff` giving the generator rank.


In [3]:
candidate_summary = (
    pl.scan_parquet(GENERATED_VALIDATION)
    .select(
        pl.len().alias("rows"),
        pl.col("impression_uid").n_unique().alias("impressions"),
        pl.col("news_code").n_unique().alias("articles"),
    )
    .collect()
)
display(candidate_summary)
pl.read_parquet(GENERATED_VALIDATION).head(5)


rows,impressions,articles
u32,u32,u32
14230800,31624,35967


impression_uid,cutoff,news_code
i64,i16,i32
10154218,1,7821
10154218,2,42977
10154218,3,12029
10154218,4,52723
10154218,5,33885


# 2. Experiment Configuration

Only the final `RR_4` architecture and loss are retrained. `TRAIN_IMPRESSIONS` and
`VALIDATION_IMPRESSIONS` keep replay laptop-sized; set either to `None` for the full split.


In [4]:
SEED = 4222
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

# Generated candidate data
TRAIN_IMPRESSIONS = 5_000          # sampled whole training impressions
VALIDATION_IMPRESSIONS = None    # None = every validation impression
# Optional output folder. None keeps the legacy branch_interaction_generated path.
# Set e.g. "train10000" together with TRAIN_IMPRESSIONS = 10_000 to keep a separate
# checkpoint, preprocessor, replay cache and score files for another sample size.
RUN_TAG = "niche"
HOLDOUT_FRACTION = 0.10            # of usable generated training impressions
MAX_CANDIDATES = 100               # every retrieved click kept, negatives sampled
HISTORY_CAP = 30                   # newest clicks kept per user
SHORT_TERM_CAP = 10                # newest clicks read by the short-term GRU

# Model (must match RR_4)
MODEL_DIM = 64
ATTENTION_HEADS = 2
ATTENTION_DROPOUT = 0.10
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.20

# Final model selected in RR_4: "+metadata" with softmax + LambdaNDCG
FINAL_CONFIG = {
    "tabular": True,
    "text": True,
    "metadata": True,
    "title_encoder": False,
    "user_encoder": True,
    "position": False,
    "candidate_pooling": False,
    "short_term": False,
}
FINAL_LOSS = "softmax+lambda_ndcg"

# Training
EPOCHS = 20
BATCH_IMPRESSIONS = 64
SCORING_BATCH_ROWS = 2048
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NDCG_K = 10
PATIENCE = 3
RESUME = True

BASE_OUTPUT_ROOT = ARTIFACTS / "branch_interaction_generated"
OUTPUT_ROOT = BASE_OUTPUT_ROOT / RUN_TAG if RUN_TAG else BASE_OUTPUT_ROOT
REPLAY_OUTPUT = OUTPUT_ROOT / "replayed_features"
RESULTS.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPLAY_OUTPUT.mkdir(parents=True, exist_ok=True)
print("Run tag:", RUN_TAG)
print("Output root:", OUTPUT_ROOT)


Using cpu
Run tag: niche
Output root: c:\Users\looyu\OneDrive\Desktop\BT4222\project\BT4222-Group-6-\5_Models\5_02_Reranking\artifacts\branch_interaction_generated\niche


## 2.1 Feature Schema

The replayed features must match the schema the reranker was trained with in `RR_4`.


In [5]:
schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
CATEGORICAL = [name for name in schema.get("categorical_columns", []) if name in FEATURES]
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
if feature_schema(DEFAULT_SETTINGS)["model_columns"] != FEATURES:
    raise ValueError("The feature code and the saved feature run disagree on the schema")
print(f"{len(FEATURES)} tabular features: {len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical")


50 tabular features: 48 numeric, 2 categorical


# 3. Prepare Generated Data

## 3.1 Load the Generator's Training Candidates

Whole impressions are sampled so listwise losses compare candidates within one request.


In [6]:
def load_generated_candidates(path, max_impressions=None):
    """Read generated candidates as an impression/news/cutoff table."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Generated candidate file not found: {path}")
    candidates = pl.read_parquet(path, columns=["impression_uid", "cutoff", "news_code"])
    if candidates.select(["impression_uid", "news_code"]).n_unique() != candidates.height:
        raise ValueError("Generated candidates contain duplicate impression/news pairs")

    impression_ids = candidates["impression_uid"].unique().sort().to_numpy()
    if max_impressions is not None:
        chosen = np.random.default_rng(SEED).choice(
            impression_ids,
            size=min(max_impressions, len(impression_ids)),
            replace=False,
        )
        candidates = candidates.filter(pl.col("impression_uid").is_in(chosen.tolist()))

    return candidates.sort(["impression_uid", "cutoff"]).to_pandas().reset_index(drop=True)


train_candidates = load_generated_candidates(GENERATED_TRAIN, TRAIN_IMPRESSIONS)
print(
    f"Train candidates: {train_candidates['impression_uid'].nunique():,} impressions / "
    f"{len(train_candidates):,} rows"
)


Train candidates: 5,000 impressions / 2,250,000 rows


## 3.2 Label and Cap Generated Slates

Clicks are joined from `labels_train`. A generated candidate that was never shown has no
label, so it becomes `0` (negative). Impressions with no retrieved click or no retrieved
non-click are dropped because the listwise losses need both. The existing slate cap keeps
every click and samples negatives to `MAX_CANDIDATES`, keeping replay small.


In [7]:
def label_generated_candidates(candidates, split):
    """Join known clicks; unshown generated candidates become 0."""
    label_files = sorted((RUN / f"labels_{split}").glob("*.parquet"))
    if not label_files:
        raise FileNotFoundError(f"No labels_{split} partitions in {RUN}")
    uids = [int(uid) for uid in candidates["impression_uid"].unique()]
    labels = (
        pl.scan_parquet(label_files)
        .filter(pl.col("impression_uid").is_in(uids))
        .select("impression_uid", "news_code", "clicked")
        .collect()
        .to_pandas()
    )
    labeled = candidates.merge(
        labels,
        on=["impression_uid", "news_code"],
        how="left",
        validate="one_to_one",
    )
    if labeled["impression_uid"].nunique() != candidates["impression_uid"].nunique():
        raise ValueError("Some generated training impressions have no labels")
    labeled["clicked"] = labeled["clicked"].fillna(0).astype(np.int8)
    return labeled


def cap_slates(frame, max_candidates):
    """Keep every click and at most `max_candidates` rows per impression (negatives sampled)."""
    rng = np.random.default_rng(SEED)
    keep = []
    for rows in frame.groupby("impression_uid", sort=False).indices.values():
        if len(rows) <= max_candidates:
            keep.append(rows)
            continue
        clicked = rows[frame["clicked"].to_numpy()[rows] == 1]
        unclicked = rows[frame["clicked"].to_numpy()[rows] == 0]
        n_negatives = max(max_candidates - len(clicked), 1)
        keep.append(
            np.concatenate(
                [clicked, rng.choice(unclicked, size=min(n_negatives, len(unclicked)), replace=False)]
            )
        )
    return frame.iloc[np.sort(np.concatenate(keep))].reset_index(drop=True)


train_labeled = label_generated_candidates(train_candidates, "train")
slate_stats = train_labeled.groupby("impression_uid")["clicked"].agg(["sum", "size"])
usable = slate_stats.index[(slate_stats["sum"] > 0) & (slate_stats["sum"] < slate_stats["size"])]
train_labeled = train_labeled[train_labeled["impression_uid"].isin(usable)].reset_index(drop=True)
if not len(train_labeled):
    raise ValueError("No generated training impression has both a retrieved click and a non-click")

train_slates = cap_slates(train_labeled, MAX_CANDIDATES)
print(
    f"Usable generated train slates: {train_slates['impression_uid'].nunique():,} impressions / "
    f"{len(train_slates):,} capped rows (click rate {train_slates['clicked'].mean():.4f})"
)


Usable generated train slates: 4,428 impressions / 442,800 capped rows (click rate 0.0139)


## 3.3 Replay Causal Tabular Features

Most generated candidates have no row in `ranking_features_*`. `build_features` is called
at each request timestamp with the state before that request, exactly as in
`PE_two_stage_pipeline.ipynb`. Features are cached so re-runs and epochs reuse them.


In [8]:
def replay_candidate_features(split, candidates):
    """Replay the causal timeline and cache the 50 tabular features for generated pairs."""
    impression_count = candidates["impression_uid"].nunique()
    cache_path = (
        REPLAY_OUTPUT
        / f"{split}_top450_{impression_count}imp_{len(candidates)}rows_replay.parquet"
    )
    if RESUME and cache_path.exists():
        print(f"Reusing replayed features: {cache_path.name}")
        return cache_path

    candidates = candidates.sort_values(["impression_uid", "cutoff"]).reset_index(drop=True)
    rows_by_impression = candidates.groupby("impression_uid", sort=False).indices
    target_ids = set(rows_by_impression)
    replay_splits = tuple(("train", "validation", "test")[: ("train", "validation", "test").index(split) + 1])

    bundle = Bundle(REPLAY_ROOT, DEFAULT_SETTINGS)
    state = TemporalState(bundle, *time_horizon(REPLAY_ROOT), DEFAULT_SETTINGS)
    partial_path = cache_path.with_suffix(".partial.parquet")
    writer, pending, impressions, started = None, [], 0, time.monotonic()

    def flush():
        nonlocal writer, pending
        if not pending:
            return
        frame = pd.concat(pending, ignore_index=True)
        frame = frame[["impression_uid", "news_code", "cutoff", *FEATURES]]
        table = pa.Table.from_pandas(frame, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(partial_path, table.schema, compression="zstd")
        writer.write_table(table)
        pending = []

    print(
        f"Replaying {split}: {len(candidates):,} candidates for "
        f"{len(rows_by_impression):,} impressions"
    )
    try:
        for timestamp, requests, events in replay_batches(
            REPLAY_ROOT, DEFAULT_SETTINGS, splits=replay_splits
        ):
            state.begin(requests, timestamp)
            selected = requests[
                requests["split"].eq(split)
                & requests["impression_uid"].isin(target_ids)
            ]
            if len(selected):
                pairs = candidates.iloc[
                    np.concatenate(
                        [rows_by_impression[uid] for uid in selected["impression_uid"]]
                    )
                ]
                features = build_features(
                    selected, pairs, state, bundle, DEFAULT_SETTINGS, "retrieved"
                )
                pending.append(
                    features.merge(
                        pairs,
                        on=["impression_uid", "news_code"],
                        validate="one_to_one",
                    )
                )
                impressions += len(selected)
                if sum(len(frame) for frame in pending) >= 50_000:
                    flush()
                    print(
                        f"  {impressions:,}/{len(rows_by_impression):,} impressions "
                        f"({(time.monotonic() - started) / 60:.1f} min)",
                        flush=True,
                    )
            state.finish(requests, events, timestamp, set(replay_splits))
        flush()
    finally:
        if writer is not None:
            writer.close()
        state.close()

    if impressions != len(rows_by_impression):
        raise ValueError(
            f"Replayed {impressions:,} impressions but {len(rows_by_impression):,} were selected"
        )
    partial_path.replace(cache_path)
    print(f"Saved replayed features: {cache_path.name}")
    return cache_path


In [9]:
train_cache = replay_candidate_features("train", train_slates)
train_replayed = (
    pl.read_parquet(train_cache)
    .select(["impression_uid", "news_code", *FEATURES])
    .to_pandas()
)
train = train_slates.merge(
    train_replayed,
    on=["impression_uid", "news_code"],
    how="left",
    validate="one_to_one",
)
if train[FEATURES].isna().any().any():
    raise ValueError("Some generated training candidates are missing replayed features")
train = train.sort_values(["impression_uid", "news_code"]).reset_index(drop=True)

impression_ids = train["impression_uid"].drop_duplicates().to_numpy()
holdout_ids = np.random.default_rng(SEED).choice(
    impression_ids,
    size=int(HOLDOUT_FRACTION * len(impression_ids)),
    replace=False,
)
holdout = train[train["impression_uid"].isin(holdout_ids)].reset_index(drop=True)
fit = train[~train["impression_uid"].isin(holdout_ids)].reset_index(drop=True)
print(
    f"Fit: {fit['impression_uid'].nunique():,} impressions / {len(fit):,} candidates; "
    f"holdout: {holdout['impression_uid'].nunique():,} impressions"
)


Replaying train: 442,800 candidates for 4,428 impressions
  500/4,428 impressions (2.1 min)
  1,000/4,428 impressions (4.4 min)
  1,500/4,428 impressions (7.1 min)
  2,000/4,428 impressions (10.5 min)
  2,500/4,428 impressions (13.8 min)
  3,000/4,428 impressions (17.6 min)
  3,500/4,428 impressions (19.5 min)
  4,000/4,428 impressions (21.0 min)
Saved replayed features: train_top450_4428imp_442800rows_replay.parquet
Fit: 3,986 impressions / 398,600 candidates; holdout: 442 impressions


## 3.4 Article Inputs

All article inputs are indexed by `article_row`; row 0 is padding.


In [10]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["news_code", "article_row", "category_code", "subcategory_code"],
)
news_to_row = dict(zip(article_index["news_code"].to_list(), article_index["article_row"].to_list()))

text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
entity_vectors = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")
title_tokens = np.load(REPRESENTATIONS / "title_token_ids.npy").astype(np.int64)
N_ARTICLES = text_vectors.shape[0]

# Category and subcategory codes per article row (0 = padding / unknown)
article_category = np.zeros(N_ARTICLES, dtype=np.int64)
article_subcategory = np.zeros(N_ARTICLES, dtype=np.int64)
article_category[article_index["article_row"].to_numpy()] = article_index["category_code"].to_numpy()
article_subcategory[article_index["article_row"].to_numpy()] = article_index["subcategory_code"].to_numpy()
print(f"{N_ARTICLES:,} article rows; {article_category.max()} categories; "
      f"{article_subcategory.max()} subcategories; title vocabulary {title_tokens.max() + 1:,}")

51,283 article rows; 17 categories; 257 subcategories; title vocabulary 28,714


## 3.5 Causal Click Histories

Histories hold the article rows clicked strictly before each impression, newest
`HISTORY_CAP` only, left-padded so the newest click is the last column.


In [11]:
def load_profiles(split, impression_ids=None):
    """Map each impression to the article rows its user clicked strictly before the impression."""
    profiles = (
        pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet")
        .select("impression_uid", "full_history_rows")
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list()))

In [12]:
def encode_histories(impression_ids, profiles):
    """Left-padded [impressions, HISTORY_CAP] article rows with the newest click last."""
    histories = np.zeros((len(impression_ids), HISTORY_CAP), dtype=np.int64)
    for index, uid in enumerate(impression_ids):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history):] = history
    return histories

## 3.6 Tabular Preprocessing

Fitted on the generated fit slates only, then saved under the generated output folder so the
logged-slate `RR_4` preprocessor is not overwritten.


In [13]:
def build_preprocessor():
    """Median-impute and standardise numeric inputs; mode-impute and one-hot encode categoricals."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC),
        ("categorical", categorical, CATEGORICAL),
    ])

In [14]:
preprocessor = build_preprocessor().fit(fit[FEATURES])
joblib.dump(preprocessor, OUTPUT_ROOT / "preprocessor_generated.joblib")
TABULAR_WIDTH = preprocessor.transform(fit[FEATURES].head(1)).shape[1]
print(f"Tabular width: {TABULAR_WIDTH}")


Tabular width: 285


## 3.7 Impression Batches

Training and the holdout check work on padded impression batches:
`rows` and `mask` are `[impressions, candidates]`, and each impression's history is encoded
once and shared by all of its candidates.


In [15]:
def build_impression_data(frame, split="train", precompute_tabular=True):
    """Arrays for padded impression batches over a training frame.

    With `precompute_tabular=False` the raw features are kept and preprocessed one batch at
    a time, which keeps memory flat when training on the whole training set.
    """
    uids = frame["impression_uid"].drop_duplicates().tolist()
    candidate_rows = frame["news_code"].map(news_to_row)
    if candidate_rows.isna().any():
        raise ValueError("A candidate is missing its article row")
    return {
        "groups": [rows.astype(np.int64) for rows in frame.groupby("impression_uid", sort=False).indices.values()],
        "histories": torch.from_numpy(encode_histories(uids, load_profiles(split, uids))),
        "tabular": (torch.from_numpy(preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False))
                    if precompute_tabular else None),
        "features": None if precompute_tabular else frame[FEATURES].reset_index(drop=True),
        "candidates": torch.from_numpy(candidate_rows.to_numpy(np.int64)),
        "labels": torch.from_numpy(frame["clicked"].to_numpy(np.float32)),
    }


def impression_batches(data, order):
    """Yield (tabular, candidates, histories, labels, mask) for batches of impressions."""
    for start in range(0, len(order), BATCH_IMPRESSIONS):
        batch = order[start:start + BATCH_IMPRESSIONS]
        groups = [data["groups"][index] for index in batch]
        width = max(len(group) for group in groups)
        rows = torch.zeros((len(batch), width), dtype=torch.long)
        mask = torch.zeros((len(batch), width), dtype=torch.bool)
        for position, group in enumerate(groups):
            rows[position, :len(group)] = torch.from_numpy(group)
            mask[position, :len(group)] = True
        if data["tabular"] is not None:
            tabular = data["tabular"][rows]
        else:
            raw = data["features"].iloc[rows.reshape(-1).numpy()]
            tabular = torch.from_numpy(preprocessor.transform(raw).astype(np.float32, copy=False)).reshape(*rows.shape, -1)
        yield (
            tabular.to(DEVICE),
            data["candidates"][rows].to(DEVICE),
            data["histories"][torch.as_tensor(batch)].to(DEVICE),
            (data["labels"][rows] * mask).to(DEVICE),   # padding never counts as a click
            mask.to(DEVICE),
        )



In [16]:
fit_data = build_impression_data(fit, split="train", precompute_tabular=False)
holdout_data = build_impression_data(holdout, split="train", precompute_tabular=False)


# 4. Model

## 4.1 Additive Attention

Scores each element with a learned query vector and returns the weighted sum. Used to
pool title words into an article vector and clicked articles into a user vector.

In [17]:
class AdditiveAttention(nn.Module):
    """Pool a sequence of history states into one user vector with learned relevance weights."""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.query = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, states, padding_mask):
        # states: [batch, history, dim]; padding_mask: [batch, history], True = padding
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False                       # softmax needs one open slot

        scores = torch.tanh(self.proj(states)) @ self.query    # [batch, history]
        weights = torch.softmax(scores.masked_fill(safe_mask, -1e9), dim=1)
        weights = weights.masked_fill(padding_mask, 0.0)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        return (weights.unsqueeze(-1) * states).sum(dim=1)     # [batch, dim]

## 4.2 `NewsEncoder`

```text
text64 (frozen) ─────────────────────────────┐
category + subcategory embeddings (metadata) ┤
entity100 (frozen) → Linear (metadata) ──────┼→ concat → Linear → LayerNorm → news vector
title words → self-attention → additive  ────┘   (title encoder)
```

The frozen article tables are stored as buffers, so a checkpoint carries no copy of them.

In [18]:
class NewsEncoder(nn.Module):
    """Map article rows to MODEL_DIM news vectors from the enabled article inputs."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.register_buffer("text", torch.from_numpy(np.array(text_vectors, dtype=np.float32)), persistent=False)
        width = self.text.shape[1]

        if config["metadata"]:
            self.register_buffer("entity", torch.from_numpy(np.array(entity_vectors, dtype=np.float32)), persistent=False)
            self.register_buffer("category", torch.from_numpy(article_category), persistent=False)
            self.register_buffer("subcategory", torch.from_numpy(article_subcategory), persistent=False)
            self.category_embedding = nn.Embedding(int(article_category.max()) + 1, 16, padding_idx=0)
            self.subcategory_embedding = nn.Embedding(int(article_subcategory.max()) + 1, 16, padding_idx=0)
            self.entity_projection = nn.Linear(self.entity.shape[1], 32)
            width += 16 + 16 + 32

        if config["title_encoder"]:
            self.register_buffer("title_tokens", torch.from_numpy(title_tokens), persistent=False)
            self.word_embedding = nn.Embedding(int(title_tokens.max()) + 1, MODEL_DIM, padding_idx=0)
            self.word_attention = nn.MultiheadAttention(MODEL_DIM, ATTENTION_HEADS, dropout=ATTENTION_DROPOUT, batch_first=True)
            self.word_pool = AdditiveAttention(MODEL_DIM)
            width += MODEL_DIM

        self.output = nn.Sequential(nn.Linear(width, MODEL_DIM), nn.LayerNorm(MODEL_DIM))

    def encode_titles(self, rows):
        tokens = self.title_tokens[rows]                       # [articles, title_len]
        padding = tokens.eq(0)
        padding[padding.all(dim=1), 0] = False                 # padding articles keep one slot
        words = self.word_embedding(tokens)
        attended, _ = self.word_attention(words, words, words, key_padding_mask=padding, need_weights=False)
        return self.word_pool(attended, padding)

    def forward(self, rows):
        # Encode each distinct article in the batch once, then scatter back to every position
        unique_rows, inverse = torch.unique(rows, return_inverse=True)
        parts = [self.text[unique_rows]]
        if self.config["metadata"]:
            parts += [
                self.category_embedding(self.category[unique_rows]),
                self.subcategory_embedding(self.subcategory[unique_rows]),
                self.entity_projection(self.entity[unique_rows]),
            ]
        if self.config["title_encoder"]:
            parts.append(self.encode_titles(unique_rows))
        return self.output(torch.cat(parts, dim=-1))[inverse]   # rows.shape + [MODEL_DIM]

## 4.3 `UserEncoder`

```text
history news vectors (+ recency position embeddings)
  → multi-head self-attention → residual + LayerNorm → feed-forward → residual + LayerNorm
  → additive attention ─────────────────→ long-term vector u_long ─┐
  → GRU over newest clicks (short_term) → u_short → gate g ────────┴→ g·u_long + (1−g)·u_short
  → candidate-as-query attention (candidate_pooling) → u_candidate (one per candidate)
```

Users with no history get a learned default vector (and a zero candidate-aware vector).
`last_gate` keeps the average gate value from the latest batch, for inspection.

In [19]:
class UserEncoder(nn.Module):
    """Encode the click history into per-candidate user vectors."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        if config["position"]:
            self.position_embedding = nn.Embedding(HISTORY_CAP, MODEL_DIM)
        self.self_attention = nn.MultiheadAttention(MODEL_DIM, ATTENTION_HEADS, dropout=ATTENTION_DROPOUT, batch_first=True)
        self.norm1 = nn.LayerNorm(MODEL_DIM)
        self.feed_forward = nn.Sequential(
            nn.Linear(MODEL_DIM, 4 * MODEL_DIM),
            nn.ReLU(),
            nn.Dropout(ATTENTION_DROPOUT),
            nn.Linear(4 * MODEL_DIM, MODEL_DIM),
        )
        self.norm2 = nn.LayerNorm(MODEL_DIM)
        self.pool = AdditiveAttention(MODEL_DIM)
        self.default_user = nn.Parameter(torch.zeros(MODEL_DIM))
        if config["short_term"]:
            self.short_term = nn.GRU(MODEL_DIM, MODEL_DIM, batch_first=True)
            self.gate = nn.Linear(2 * MODEL_DIM, MODEL_DIM)
        self.last_gate = None

    @property
    def output_dim(self):
        return MODEL_DIM * (2 if self.config["candidate_pooling"] else 1)

    def forward(self, history, padding_mask, candidates):
        # history: [B, H, D]; padding_mask: [B, H] (True = padding); candidates: [B, W, D]
        empty = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[empty, -1] = False

        if self.config["position"]:
            # Column j of a left-padded history has recency rank H-1-j (0 = newest)
            ranks = torch.arange(HISTORY_CAP - 1, -1, -1, device=history.device)
            history = history + self.position_embedding(ranks)

        attended, _ = self.self_attention(history, history, history, key_padding_mask=safe_mask, need_weights=False)
        states = self.norm1(history + attended)
        states = self.norm2(states + self.feed_forward(states))
        states = states.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        user = self.pool(states, padding_mask)
        if self.config["short_term"]:
            recent = states[:, -SHORT_TERM_CAP:]                       # newest clicks, zeros for padding
            _, short = self.short_term(recent)
            short = short.squeeze(0)
            gate = torch.sigmoid(self.gate(torch.cat([user, short], dim=1)))
            user = gate * user + (1 - gate) * short
            self.last_gate = gate.detach().mean().item()
        user = torch.where(empty.unsqueeze(1), self.default_user.unsqueeze(0), user)
        user = user.unsqueeze(1).expand(-1, candidates.shape[1], -1)   # [B, W, D]

        if not self.config["candidate_pooling"]:
            return user
        # Candidate-as-query attention over the contextualised history
        scores = candidates @ states.transpose(1, 2) / np.sqrt(MODEL_DIM)       # [B, W, H]
        scores = scores.masked_fill(safe_mask.unsqueeze(1), -1e9)
        weights = torch.softmax(scores, dim=-1).masked_fill(padding_mask.unsqueeze(1), 0.0)
        candidate_user = weights @ states                                       # [B, W, D]
        return torch.cat([user, candidate_user], dim=-1)

## 4.4 `BranchReranker`

```text
[tabular (tabular) ; candidate news vector (text) ; user vectors (user_encoder)]
    → Dense 256 → ReLU → Dropout → Dense 64 → ReLU → Dropout → logit
```

Every architecture in the grid is this class with different switches.

In [20]:
class BranchReranker(nn.Module):
    """Score candidates from the enabled tabular, news and user branches."""

    def __init__(self, config, tabular_width):
        super().__init__()
        if config["user_encoder"] and not config["text"]:
            raise ValueError("The user encoder needs the news encoder (text=True)")
        if not (config["tabular"] or config["text"]):
            raise ValueError("At least one of the tabular or text branches must be on")
        self.config = config
        width = tabular_width if config["tabular"] else 0
        if config["text"]:
            self.news_encoder = NewsEncoder(config)
            width += MODEL_DIM
        if config["user_encoder"]:
            self.user_encoder = UserEncoder(config)
            width += self.user_encoder.output_dim

        layers = []
        for hidden in HIDDEN_DIMS:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(DROPOUT)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.scorer = nn.Sequential(*layers)

    def forward(self, tabular, candidate_rows, history_rows):
        # tabular: [B, W, T]; candidate_rows: [B, W]; history_rows: [B, H] → logits [B, W]
        parts = [tabular] if self.config["tabular"] else []
        if self.config["text"]:
            candidates = self.news_encoder(candidate_rows)
            parts.append(candidates)
            if self.config["user_encoder"]:
                history = self.news_encoder(history_rows)
                parts.append(self.user_encoder(history, history_rows.eq(0), candidates))
        return self.scorer(torch.cat(parts, dim=-1)).squeeze(-1)

# 5. Loss Functions

All three take `[impressions, candidates]` logits, labels and a padding mask.

- **Listwise softmax**: for each click, cross-entropy of that click against every unclicked
  candidate in its impression: $-\log \frac{e^{s^+}}{e^{s^+} + \sum_{j \in neg} e^{s_j}}$.
  It spreads probability over the whole slate and favours MRR-style "click on top" behaviour.
- **LambdaNDCG@K**: a RankNet loss on (click, non-click) pairs weighted by how much swapping
  them would change nDCG@K, so it concentrates on the top-K order the metric reads.
- **Weighted BCE**: each candidate on its own, with a positive-class weight from the
  training click rate. The pointwise reference used by every earlier notebook.
- **BPR**: $-\log\sigma(s^+ - s^-)$ averaged over every (click, non-click) pair in the impression.
- **Combinations**: weighted sums such as `softmax+bce` (0.8 / 0.2). The losses have different
  scales, so the weights set the balance rather than a strict percentage.

In [21]:
def listwise_softmax_loss(logits, labels, mask):
    """Mean over clicks of -log softmax(click vs. all unclicked candidates in its impression)."""
    positive = (labels > 0) & mask
    negative = (labels == 0) & mask
    negative_lse = torch.logsumexp(logits.masked_fill(~negative, float("-inf")), dim=1, keepdim=True)
    log_probability = logits - torch.logaddexp(logits, negative_lse)
    return -(log_probability * positive).sum() / positive.sum().clamp_min(1)


def lambda_ndcg_loss(logits, labels, mask, k=NDCG_K):
    """RankNet loss weighted by detached |ΔnDCG@k| for every (click, non-click) pair."""
    masked_logits = logits.masked_fill(~mask, float("-inf"))
    ranks = masked_logits.argsort(dim=1, descending=True).argsort(dim=1).add(1).detach()
    gains = (torch.pow(2.0, labels) - 1.0) * mask
    discounts = (1.0 / torch.log2(ranks.float() + 1.0)) * (ranks <= k) * mask

    ideal_labels = labels.masked_fill(~mask, float("-inf")).sort(dim=1, descending=True).values
    ideal_mask = torch.isfinite(ideal_labels)
    ideal_labels = ideal_labels.clamp_min(0)
    positions = torch.arange(1, logits.shape[1] + 1, device=logits.device)
    ideal_discounts = (1.0 / torch.log2(positions.float() + 1.0)) * (positions <= k)
    idcg = ((torch.pow(2.0, ideal_labels) - 1.0) * ideal_discounts * ideal_mask).sum(dim=1).clamp_min(1e-12)

    score_diff = logits.unsqueeze(2) - logits.unsqueeze(1)
    delta_ndcg = ((gains.unsqueeze(2) - gains.unsqueeze(1)) * (discounts.unsqueeze(2) - discounts.unsqueeze(1))).abs()
    weights = delta_ndcg / idcg[:, None, None]
    valid_pairs = (labels.unsqueeze(2) > labels.unsqueeze(1)) & mask.unsqueeze(2) & mask.unsqueeze(1)
    weights = weights * valid_pairs
    return (F.softplus(-score_diff) * weights).sum() / weights.sum().clamp_min(1e-12)


CLICK_RATE = fit["clicked"].mean()
POSITIVE_WEIGHT = torch.tensor(float((1 - CLICK_RATE) / CLICK_RATE), dtype=torch.float32, device=DEVICE)


def weighted_bce_loss(logits, labels, mask):
    """Pointwise BCE over real candidates with a training-derived positive weight."""
    loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=POSITIVE_WEIGHT, reduction="none")
    return (loss * mask).sum() / mask.sum()


def bpr_loss(logits, labels, mask):
    """Mean -log σ(s⁺ − s⁻) over every (click, non-click) pair within each impression."""
    positive = (labels > 0) & mask
    negative = (labels == 0) & mask
    pairs = positive.unsqueeze(2) & negative.unsqueeze(1)              # [B, W, W]: i clicked, j not
    score_diff = logits.unsqueeze(2) - logits.unsqueeze(1)
    return (F.softplus(-score_diff) * pairs).sum() / pairs.sum().clamp_min(1)


BASE_LOSSES = {
    "softmax": listwise_softmax_loss,
    "lambda_ndcg": lambda_ndcg_loss,
    "bce": weighted_bce_loss,
    "bpr": bpr_loss,
}


def combined_loss(**weights):
    """Weighted sum of base losses, e.g. combined_loss(softmax=0.8, bce=0.2)."""
    def loss(logits, labels, mask):
        return sum(weight * BASE_LOSSES[name](logits, labels, mask) for name, weight in weights.items())
    return loss


# Combinations pair a ranking loss with a second objective:
#   + bce          keeps scores calibrated as click probabilities
#   + lambda_ndcg  adds top-K focus to the whole-slate softmax
LOSSES = {
    **BASE_LOSSES,
    "softmax+bce": combined_loss(softmax=0.8, bce=0.2),
    "softmax+lambda_ndcg": combined_loss(softmax=0.5, lambda_ndcg=0.5),
    "lambda_ndcg+bce": combined_loss(lambda_ndcg=0.8, bce=0.2),
    "bpr+bce": combined_loss(bpr=0.75, bce=0.25),
}

# 6. Training and Scoring

## 6.1 Training Loop

Each epoch shuffles the fit impressions. After every epoch the model ranks the
holdout impressions' retained candidates; the epoch with the best holdout nDCG@10 is kept.
This uses training data only, so validation is untouched until scoring. With `patience`
set, training also stops early once the holdout score stops improving.

In [22]:
def batch_ndcg(logits, labels, mask, k=10):
    """Mean nDCG@k over a batch of impressions (retained candidates only)."""
    order = logits.masked_fill(~mask, float("-inf")).argsort(dim=1, descending=True)[:, :k]
    discounts = 1.0 / torch.log2(torch.arange(2, order.shape[1] + 2, device=logits.device).float())
    dcg = (labels.gather(1, order) * discounts).sum(dim=1)
    ideal = (labels.masked_fill(~mask, 0).sort(dim=1, descending=True).values[:, :order.shape[1]] * discounts).sum(dim=1)
    return (dcg / ideal.clamp_min(1e-12)).mean().item()


def train_model(config, loss_name, fit_data=None, holdout_data=None, epochs=EPOCHS, patience=None):
    """Train one architecture with one loss; return the best-holdout-epoch model.

    Defaults train on the 10k-impression sample. `patience` stops training after that many
    epochs without a holdout nDCG@10 improvement.
    """
    fit_data = fit_data if fit_data is not None else globals()["fit_data"]
    holdout_data = holdout_data if holdout_data is not None else globals()["holdout_data"]
    torch.manual_seed(SEED)
    model = BranchReranker(config, TABULAR_WIDTH).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = LOSSES[loss_name]
    rng = np.random.default_rng(SEED)
    best = {"ndcg": -1.0, "epoch": 0, "state": None}

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for tabular, candidates, histories, labels, mask in impression_batches(fit_data, rng.permutation(len(fit_data["groups"]))):
            optimizer.zero_grad()
            loss = loss_fn(model(tabular, candidates, histories), labels, mask)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            holdout_scores = [
                batch_ndcg(model(tabular, candidates, histories), labels, mask)
                for tabular, candidates, histories, labels, mask in impression_batches(holdout_data, np.arange(len(holdout_data["groups"])))
            ]
        holdout_ndcg = float(np.mean(holdout_scores))
        print(f"  epoch {epoch}/{epochs}: {loss_name} loss {np.mean(losses):.4f}, holdout nDCG@10 {holdout_ndcg:.4f}")
        if holdout_ndcg > best["ndcg"]:
            best = {"ndcg": holdout_ndcg, "epoch": epoch,
                    "state": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}}
        elif patience is not None and epoch - best["epoch"] >= patience:
            print(f"  early stopping: no holdout improvement for {patience} epochs; keeping epoch {best['epoch']}")
            break

    model.load_state_dict(best["state"])
    model.best_epoch, model.best_holdout_ndcg = best["epoch"], best["ndcg"]
    return model

## 6.2 Scoring Generated Candidates

Scoring is pointwise, so cached feature rows are streamed in fixed-size chunks. The score
file keeps `cutoff` so the generator's own order can be evaluated as a baseline.


In [23]:
def branch_predictor(model, profiles):
    """Wrap a trained model and a split's histories as a predict(frame) function."""

    def predict(frame):
        uids = frame["impression_uid"].drop_duplicates().tolist()
        histories = encode_histories(uids, profiles)
        history_index = frame["impression_uid"].map(
            pd.Series(np.arange(len(uids)), index=uids)
        ).to_numpy()
        tabular = preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
        candidates = frame["news_code"].map(news_to_row).to_numpy(np.int64)

        model.eval()
        scores = []
        with torch.no_grad():
            for start in range(0, len(frame), SCORING_BATCH_ROWS):
                stop = start + SCORING_BATCH_ROWS
                logits = model(
                    torch.from_numpy(tabular[start:stop]).unsqueeze(1).to(DEVICE),
                    torch.from_numpy(candidates[start:stop]).unsqueeze(1).to(DEVICE),
                    torch.from_numpy(histories[history_index[start:stop]]).to(DEVICE),
                )
                scores.append(torch.sigmoid(logits.squeeze(1)).cpu().numpy())
        return np.concatenate(scores)

    return predict


def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = BranchReranker(checkpoint["config"], checkpoint["tabular_width"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model


def score_feature_cache(predict, cache_path, score_path):
    """Stream a replayed-feature cache through `predict` and write a score file."""
    parquet = pq.ParquetFile(cache_path)
    key_parts, score_parts = [], []
    for batch in parquet.iter_batches(batch_size=SCORING_BATCH_ROWS):
        frame = batch.to_pandas()
        key_parts.append(frame[["impression_uid", "news_code", "cutoff"]])
        score_parts.append(predict(frame))
    scored = pd.concat(key_parts, ignore_index=True)
    scored["score"] = np.concatenate(score_parts)
    scored = pl.from_pandas(scored)
    scored.write_parquet(score_path)
    return scored


# 7. Final Training on Generated Candidates

The `+metadata` architecture is trained with `softmax+lambda_ndcg` on the generated
training slates. The holdout is 10% of the usable generated training impressions; early
stopping uses holdout nDCG@10.


In [24]:
FINAL_NAME = "branch_plus_metadata__softmax_plus_lambda_ndcg__generated"
final_dir = OUTPUT_ROOT / FINAL_NAME
final_dir.mkdir(parents=True, exist_ok=True)

if RESUME and (final_dir / "model.pt").exists():
    print(f"already trained: {FINAL_NAME}")
    final_model = load_checkpoint(final_dir / "model.pt")
else:
    final_model = train_model(
        FINAL_CONFIG,
        FINAL_LOSS,
        fit_data=fit_data,
        holdout_data=holdout_data,
        epochs=EPOCHS,
        patience=PATIENCE,
    )
    torch.save(
        {
            "state_dict": final_model.state_dict(),
            "config": FINAL_CONFIG,
            "loss": FINAL_LOSS,
            "tabular_width": TABULAR_WIDTH,
            "best_epoch": final_model.best_epoch,
            "seed": SEED,
        },
        final_dir / "model.pt",
    )
    print(
        f"Best epoch: {final_model.best_epoch} "
        f"(holdout nDCG@10 {final_model.best_holdout_ndcg:.4f})"
    )


  epoch 1/20: softmax+lambda_ndcg loss 1.8088, holdout nDCG@10 0.5143
  epoch 2/20: softmax+lambda_ndcg loss 1.5083, holdout nDCG@10 0.5387
  epoch 3/20: softmax+lambda_ndcg loss 1.4516, holdout nDCG@10 0.5527
  epoch 4/20: softmax+lambda_ndcg loss 1.4119, holdout nDCG@10 0.5573
  epoch 5/20: softmax+lambda_ndcg loss 1.3892, holdout nDCG@10 0.5687
  epoch 6/20: softmax+lambda_ndcg loss 1.3506, holdout nDCG@10 0.5738
  epoch 7/20: softmax+lambda_ndcg loss 1.3319, holdout nDCG@10 0.5841
  epoch 8/20: softmax+lambda_ndcg loss 1.3019, holdout nDCG@10 0.5831
  epoch 9/20: softmax+lambda_ndcg loss 1.2746, holdout nDCG@10 0.5882
  epoch 10/20: softmax+lambda_ndcg loss 1.2610, holdout nDCG@10 0.5887
  epoch 11/20: softmax+lambda_ndcg loss 1.2251, holdout nDCG@10 0.5832
  epoch 12/20: softmax+lambda_ndcg loss 1.1994, holdout nDCG@10 0.5766
  epoch 13/20: softmax+lambda_ndcg loss 1.1721, holdout nDCG@10 0.5805
  early stopping: no holdout improvement for 3 epochs; keeping epoch 10
Best epoch: 10

# 8. Validation on Generated Candidates

Every candidate in the generator's validation list is scored. `evaluate_retrieved_candidates`
treats unshown candidates as unknown, keeps every known click in the denominator, and
reports `candidate_recall` as the ceiling. The generator's own cutoff order is shown as a
baseline.


In [25]:
validation_candidates = load_generated_candidates(
    GENERATED_VALIDATION, VALIDATION_IMPRESSIONS
)
print(
    f"Validation candidates: {validation_candidates['impression_uid'].nunique():,} impressions / "
    f"{len(validation_candidates):,} rows"
)
validation_cache = replay_candidate_features("validation", validation_candidates)
validation_profiles = load_profiles(
    "validation", validation_candidates["impression_uid"].unique().tolist()
)
validation_scores = score_feature_cache(
    branch_predictor(final_model, validation_profiles),
    validation_cache,
    final_dir / "generated_validation_scores.parquet",
)

reranker_result = evaluate_retrieved_candidates(
    validation_scores,
    feature_run=RUN,
    split="validation",
    score_column="score",
    model_name=FINAL_NAME,
)
generator_result = evaluate_retrieved_candidates(
    validation_scores.with_columns((-pl.col("cutoff")).alias("generator_score")),
    feature_run=RUN,
    split="validation",
    score_column="generator_score",
    model_name="generator_order",
)
comparison = (
    pl.concat([reranker_result, generator_result], how="diagonal_relaxed")
    .sort("ndcg_at_10", descending=True)
)
comparison.write_csv(RESULTS / f"reranking_{FINAL_NAME}_generated_validation.csv")
display(comparison)


Validation candidates: 31,624 impressions / 14,230,800 rows
Replaying validation: 14,230,800 candidates for 31,624 impressions
  112/31,624 impressions (12.5 min)
  224/31,624 impressions (12.6 min)
  336/31,624 impressions (12.7 min)
  448/31,624 impressions (12.7 min)
  560/31,624 impressions (12.8 min)
  672/31,624 impressions (12.9 min)
  784/31,624 impressions (12.9 min)
  896/31,624 impressions (13.0 min)
  1,008/31,624 impressions (13.0 min)
  1,120/31,624 impressions (13.1 min)
  1,232/31,624 impressions (13.2 min)
  1,344/31,624 impressions (13.2 min)
  1,456/31,624 impressions (13.3 min)
  1,568/31,624 impressions (13.3 min)
  1,680/31,624 impressions (13.4 min)
  1,792/31,624 impressions (13.4 min)
  1,904/31,624 impressions (13.5 min)
  2,018/31,624 impressions (13.5 min)
  2,130/31,624 impressions (13.6 min)
  2,242/31,624 impressions (13.6 min)
  2,354/31,624 impressions (13.7 min)
  2,466/31,624 impressions (13.7 min)
  2,580/31,624 impressions (13.8 min)
  2,692/31,624 

model,split,impressions,candidate_recall,mrr,ndcg_at_5,recall_at_5,ndcg_at_10,recall_at_10
str,str,i64,f64,f64,f64,f64,f64,f64
"""generator_order""","""validation""",31624,0.808032,0.216234,0.193038,0.269076,0.223206,0.355928
"""branch_plus_metadata__softmax_…","""validation""",31624,0.808032,0.172494,0.141061,0.194881,0.170712,0.28172


# 9. Test Evaluation

Runs only when a generated test candidate file exists. Otherwise it is skipped: the model
was trained and selected on generated train/validation data, and no generated test slates
have been produced.


In [26]:
if GENERATED_TEST.exists():
    test_candidates = load_generated_candidates(GENERATED_TEST, None)
    test_cache = replay_candidate_features("test", test_candidates)
    test_profiles = load_profiles(
        "test", test_candidates["impression_uid"].unique().tolist()
    )
    test_scores = score_feature_cache(
        branch_predictor(final_model, test_profiles),
        test_cache,
        final_dir / "generated_test_scores.parquet",
    )
    test_result = evaluate_retrieved_candidates(
        test_scores,
        feature_run=RUN,
        split="test",
        score_column="score",
        model_name=FINAL_NAME,
    )
    display(test_result)
else:
    print(f"Skipped: no generated test candidates at {GENERATED_TEST}")


Replaying test: 13,621,500 candidates for 30,270 impressions
  112/30,270 impressions (18.0 min)
  224/30,270 impressions (18.0 min)
  336/30,270 impressions (18.1 min)
  448/30,270 impressions (18.2 min)
  560/30,270 impressions (18.3 min)
  672/30,270 impressions (18.3 min)
  784/30,270 impressions (18.4 min)
  897/30,270 impressions (18.5 min)
  1,009/30,270 impressions (18.5 min)
  1,121/30,270 impressions (18.6 min)
  1,233/30,270 impressions (18.6 min)
  1,345/30,270 impressions (18.7 min)
  1,457/30,270 impressions (18.8 min)
  1,570/30,270 impressions (18.8 min)
  1,682/30,270 impressions (18.9 min)
  1,794/30,270 impressions (19.0 min)
  1,906/30,270 impressions (19.0 min)
  2,018/30,270 impressions (19.1 min)
  2,130/30,270 impressions (19.1 min)
  2,242/30,270 impressions (19.2 min)
  2,355/30,270 impressions (19.3 min)
  2,467/30,270 impressions (19.3 min)
  2,579/30,270 impressions (19.4 min)
  2,691/30,270 impressions (19.5 min)
  2,803/30,270 impressions (19.5 min)
  2,9

model,split,impressions,candidate_recall,mrr,ndcg_at_5,recall_at_5,ndcg_at_10,recall_at_10
str,str,i64,f64,f64,f64,f64,f64,f64
"""branch_plus_metadata__softmax_…","""test""",30270,0.799936,0.165377,0.132037,0.179268,0.158394,0.255822


# 10. Recommendations for a User

Ranks the generator's candidates for one scored validation impression. Unlike
`rank_logged_slate`, this shows articles the generator retrieved, not only articles that
were shown.


In [27]:
def rank_generated_slate(user_code, impression_uid, score_file, candidates_file, top_n=5):
    candidates = pl.read_parquet(candidates_file).filter(
        pl.col("impression_uid") == impression_uid
    )
    scores = pl.read_parquet(score_file).filter(
        pl.col("impression_uid") == impression_uid
    )
    ranked = candidates.join(scores, on=["impression_uid", "news_code"], how="left")
    if ranked["score"].null_count():
        raise ValueError("The score file does not cover this generated slate")
    return (
        ranked.with_columns(pl.lit(user_code).alias("user_code"))
        .sort(["score", "news_code"], descending=[True, False])
        .select("user_code", "impression_uid", "news_code", "score")
        .head(top_n)
    )


example_impression = int(validation_scores["impression_uid"][0])
user_row = (
    pl.scan_parquet(RUN / "ranking_features_validation" / "*.parquet")
    .filter(pl.col("impression_uid") == example_impression)
    .select("user_code")
    .unique()
    .collect()
)
example_user = int(user_row["user_code"][0])
rank_generated_slate(
    example_user,
    example_impression,
    final_dir / "generated_validation_scores.parquet",
    GENERATED_VALIDATION,
    top_n=5,
)


user_code,impression_uid,news_code,score
i32,i64,i32,f32
11484,10154218,42977,0.902296
11484,10154218,33885,0.878568
11484,10154218,20495,0.753192
11484,10154218,52723,0.668201
11484,10154218,12029,0.665237


## Notes

- Training uses all generated candidates for the sampled impressions. Candidates that were
  never shown are treated as negatives, which can introduce false negatives.
- Validation treats unshown candidates as unknown, so `candidate_recall` is the ceiling for
  every ranking metric and clicked articles the generator missed count as misses.
- Replayed features are cached under `branch_interaction_generated/replayed_features`; delete
  a cache file to rebuild it after changing the sampled impressions or candidate file.
- Set `TRAIN_IMPRESSIONS` or `VALIDATION_IMPRESSIONS` to `None` for a full-split run.
